# 9. Linguistic and embedding fusion

## Objective

Feature fusion combines the original handcrafted linguistic measurements with the 384-dimensional transcript embedding. The intent is to let the model use both interpretable surface signals and dense contextual information.

The fused matrix is median-imputed and standardized inside Ridge, then evaluated with the same shuffled 5-fold cross-validation protocol. The recorded result determines whether fusion helps or hurts relative to the separate models.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

PROJECT_ROOT = r"C:\Users\rudra\Desktop\SHL-Grammar-Scoring"

features = pd.read_csv(
    PROJECT_ROOT + r"\outputs\linguistic_features.csv"
)

embeddings = np.load(
    PROJECT_ROOT + r"\outputs\text_embeddings.npy"
)

y = features["label"].values

X_linguistic = features.drop(
    columns=["filename", "label"]
)

constant_cols = X_linguistic.columns[
    X_linguistic.nunique() <= 1
]

X_linguistic = X_linguistic.drop(
    columns=constant_cols
)

print("Linguistic:", X_linguistic.shape)
print("Embeddings:", embeddings.shape)

In [ ]:
X_fused = np.hstack([
    X_linguistic.values,
    embeddings
])

print("Fused feature shape:", X_fused.shape)

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

fusion_pred = cross_val_predict(
    model,
    X_fused,
    y,
    cv=kf
)

fusion_rmse = np.sqrt(
    mean_squared_error(y, fusion_pred)
)

fusion_pearson = pearsonr(
    y,
    fusion_pred
)[0]

print("Linguistic + Text Embedding + Ridge")
print("RMSE:", fusion_rmse)
print("Pearson:", fusion_pearson)

In [ ]:
## Results

| Experiment | RMSE | Pearson | Decision |
|---|---:|---:|---|
| Mean baseline | 1.2382 | — | Baseline |
| **Linguistic + Ridge** | **0.9909** | **0.5997** | Keep |
| Linguistic + GBDT | 0.9954 | 0.5950 | Drop |
| Text Embedding + Ridge | 1.1836 | 0.5200 | Drop |
| Linguistic + Embedding + Ridge | 1.1654 | 0.5431 | Drop |

## Interpretation

The recorded fused embedding-plus-linguistic Ridge result is RMSE `1.165390339529917` and Pearson `0.5431054092972271`. It is weaker than the original linguistic Ridge result, so this fusion did not help under the fixed protocol. The later final pipeline instead evaluates acoustic information together with the linguistic features.